# Capstone Two: Data Wrangling

## Predictive Healthcare Operations & Capacity Analytics

This notebook prepares California hospital financial and utilization data from the
California Department of Health Care Access and Information (HCAI) for exploratory
data analysis and predictive modeling.

The data covers quarterly hospital information from 2021 Q1 through 2026 Q1.

The main data-wrangling tasks are:
- Load and combine the quarterly datasets
- Inspect the structure and data types
- Check for duplicates and missing values
- Check for invalid values and outliers
- Standardize variables where needed
- Create a clean dataset for the next stage of the project

## 1. Import Libraries

Import the Python libraries needed to load, inspect, clean, and summarize the data.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 2. Load the HCAI Data

The raw HCAI quarterly files are stored in the `data/raw/` folder.

All quarterly files will be loaded and combined into one DataFrame.

In [2]:
# Set the folder containing the raw HCAI files
data_path = Path("../data/raw")

# Find CSV files in the folder
files = sorted(data_path.glob("*.csv"))

# Display the files found
print(f"Number of files found: {len(files)}")
for file in files:
    print(file.name)

Number of files found: 21
2021_q1.csv
2021_q2.csv
2021_q3.csv
2021_q4.csv
2022_q1.csv
2022_q2.csv
2022_q3.csv
2022_q4.csv
2023_q1.csv
2023_q2.csv
2023_q3.csv
2023_q4.csv
2024_q1.csv
2024_q2.csv
2024_q3.csv
2024_q4.csv
2025_q1.csv
2025_q2.csv
2025_q3.csv
2025_q4.csv
2026_q1.csv


In [3]:
# Load each quarterly file and combine them into one DataFrame
dfs = [pd.read_csv(file) for file in files]

df = pd.concat(dfs, ignore_index=True)

print(f"Combined dataset shape: {df.shape}")

Combined dataset shape: (9179, 159)


## 3. Initial Data Inspection

Inspect the size, structure, data types, and basic statistics of the combined dataset.

In [4]:
# Display the first few records
display(df.head())

# Display number of rows and columns
print("Shape:", df.shape)

# Display data types and non-null counts
df.info()

,FAC_NO,FAC_NAME,YEAR_QTR,BEG_DATE,END_DATE,OP_STATUS,COUNTY_NAME,HSA,HFPA,TYPE_CNTRL,...,CURR_LIABILITIES,CURR_MAT_LT_DEBT,TOT_CURR_LIABILITIES,TOT_DEF_CREDITS,TOT_LT_DEBT,CURR_MATURITIES,NET_TOT_LT_DEBT,TOT_LIABILITIES,TOT_EQUITY,TOT_LIABILITY_AND EQUITY
0,106580996,ADVENTIST HEALTH AND RIDEOUT,20211,01/01/2021,03/31/2021,Open,Yuba,02 - Golden Empire,227,Non Profit Corp.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,106150788,ADVENTIST HEALTH BAKERSFIELD,20211,01/01/2021,03/31/2021,Open,Kern,09 - Central,617,Non Profit Corp.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,106171049,ADVENTIST HEALTH CLEARLAKE,20211,01/01/2021,03/31/2021,Open,Lake,01 - Northern California,115,Non Profit Corp.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,106150706,ADVENTIST HEALTH DELANO,20211,01/01/2021,03/31/2021,Open,Kern,09 - Central,617,Non Profit Corp.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,106190323,ADVENTIST HEALTH GLENDALE,20211,01/01/2021,03/31/2021,Open,Los Angeles,11 - Los Angeles,909,Church,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Shape: (9179, 159)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9179 entries, 0 to 9178
Columns: 159 entries, FAC_NO to TOT_LIABILITY_AND EQUITY
dtypes: float64(39), int64(106), object(14)
memory usage: 11.1+ MB


In [5]:
# Summary statistics for numeric variables
df.describe().T

,count,mean,std,min,25%,50%,75%,max
FAC_NO,9179.0,1.062849e+08,1.359621e+05,1.060107e+08,106190385.0,106301283.0,106374049.0,1.065810e+08
YEAR_QTR,9179.0,2.023387e+04,1.510495e+01,2.021100e+04,20222.0,20233.0,20244.0,2.026100e+04
HFPA,9179.0,7.358787e+02,3.516590e+02,1.010000e+02,421.0,807.0,935.0,1.424000e+03
LIC_BEDS,9179.0,2.195151e+02,2.235625e+02,0.000000e+00,66.5,152.0,317.5,1.500000e+03
AVL_BEDS,9179.0,2.052578e+02,2.026006e+02,0.000000e+00,62.0,148.0,291.0,1.459000e+03
...,...,...,...,...,...,...,...,...
CURR_MATURITIES,3933.0,9.954144e+06,9.259620e+07,-5.321671e+07,0.0,361611.0,2757050.0,2.268088e+09
NET_TOT_LT_DEBT,3933.0,1.528959e+08,5.374254e+08,-5.220358e+08,9457.0,10929620.0,90908858.0,8.007294e+09
TOT_LIABILITIES,3933.0,3.772026e+08,1.195524e+09,-5.079537e+08,9340785.0,67728926.0,251085806.0,1.321767e+10
TOT_EQUITY,3933.0,3.197920e+08,1.523780e+09,-1.630121e+09,0.0,33561735.0,245458374.0,2.331943e+10


## 4. Check the Reporting Period

Verify that the combined dataset contains the expected quarterly reporting periods.

In [6]:
# Check the available reporting periods
print("Number of reporting periods:", df["YEAR_QTR"].nunique())

display(
    df["YEAR_QTR"]
    .value_counts()
    .sort_index()
)

# Check the first and last reporting periods
print("First period:", df["YEAR_QTR"].min())
print("Last period:", df["YEAR_QTR"].max())

Number of reporting periods: 21


YEAR_QTR
20211    433
20212    435
20213    436
20214    436
20221    438
20222    439
20223    438
20224    437
20231    436
20232    439
20233    440
20234    439
20241    438
20242    437
20243    433
20244    436
20251    442
20252    436
20253    438
20254    437
20261    436
Name: count, dtype: int64

First period: 20211
Last period: 20261


## 5. Review Variables

Review the column names to understand the variables available for analysis.

In [7]:
# Display all column names
for i, column in enumerate(df.columns, start=1):
    print(i, column)

1 FAC_NO
2 FAC_NAME
3 YEAR_QTR
4 BEG_DATE
5 END_DATE
6 OP_STATUS
7 COUNTY_NAME
8 HSA
9 HFPA
10 TYPE_CNTRL
11 TYPE_HOSP
12 TEACH_RURL
13 PHONE
14 ADDRESS
15 CITY
16 ZIP_CODE
17 CEO
18 LIC_BEDS
19 AVL_BEDS
20 STF_BEDS
21 DIS_MCAR
22 DIS_MCAR_MC
23 DIS_MCAL
24 DIS_MCAL_MC
25 DIS_CNTY
26 DIS_CNTY_MC
27 DIS_THRD
28 DIS_THRD_MC
29 DIS_INDGNT
30 DIS_OTH
31 DIS_TOT
32 DIS_LTC
33 DAY_MCAR
34 DAY_MCAR_MC
35 DAY_MCAL
36 DAY_MCAL_MC
37 DAY_CNTY
38 DAY_CNTY_MC
39 DAY_THRD
40 DAY_THRD_MC
41 DAY_INDGNT
42 DAY_OTH
43 DAY_TOT
44 DAY_LTC
45 VIS_MCAR
46 VIS_MCAR_MC
47 VIS_MCAL
48 VIS_MCAL_MC
49 VIS_CNTY
50 VIS_CNTY_MC
51 VIS_THRD
52 VIS_THRD_MC
53 VIS_INDGNT
54 VIS_OTH
55 VIS_TOT
56 GRIP_MCAR
57 GRIP_MCAR_MC
58 GRIP_MCAL
59 GRIP_MCAL_MC
60 GRIP_CNTY
61 GRIP_CNTY_MC
62 GRIP_THRD
63 GRIP_THRD_MC
64 GRIP_INDGNT
65 GRIP_OTH
66 GRIP_TOT
67 GROP_MCAR
68 GROP_MCAR_MC
69 GROP_MCAL
70 GROP_MCAL_MC
71 GROP_CNTY
72 GROP_CNTY_MC
73 GROP_THRD
74 GROP_THRD_MC
75 GROP_INDGNT
76 GROP_OTH
77 GROP_TOT
78 BAD_DEBT
79 CADJ_

## 6. Schema Change

The HCAI files changed structure during the study period. Files from 2021 Q1
through 2023 Q4 contain 133 columns, while files from 2024 Q1 through 2026 Q1
contain 146 columns.

After combining the files, the dataset contains 159 unique columns because
some variables were added or removed over time. The resulting missing values
are retained for further investigation.

In [8]:
# Compare the number of rows and columns in each source file
schema_check = []

for file, data in zip(files, dfs):
    schema_check.append({
        "file": file.name,
        "rows": len(data),
        "columns": len(data.columns)
    })

schema_check = pd.DataFrame(schema_check)

display(schema_check)

,file,rows,columns
0,2021_q1.csv,433,133
1,2021_q2.csv,435,133
2,2021_q3.csv,436,133
3,2021_q4.csv,436,133
4,2022_q1.csv,438,133
5,2022_q2.csv,439,133
6,2022_q3.csv,438,133
7,2022_q4.csv,437,133
8,2023_q1.csv,436,133
9,2023_q2.csv,439,133


## 7. Check for Duplicate Records

Check for completely duplicated rows and duplicate facility-quarter records.

In [9]:
# Check for completely duplicated rows
duplicate_rows = df.duplicated().sum()

# Check for duplicate facility-quarter records
facility_period_duplicates = df.duplicated(
    subset=["FAC_NO", "YEAR_QTR"]
).sum()

print("Completely duplicated rows:", duplicate_rows)
print("Duplicate facility-quarter records:", facility_period_duplicates)

Completely duplicated rows: 0
Duplicate facility-quarter records: 0


## 8. Check Missing Values

Identify variables with missing values and calculate both the number and percentage
of missing observations.

In [10]:
# Calculate missing-value count and percentage
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100
})

# Display variables with missing values
missing = (
    missing[missing["missing_count"] > 0]
    .sort_values("missing_percent", ascending=False)
)

display(missing)

,missing_count,missing_percent
TEACH_RURL,7148,77.873407
CASH,5249,57.184879
TOT_ASSETS,5246,57.152195
TOT_LIMITED_USE_ASSETS,5246,57.152195
TOT_PPE,5246,57.152195
ACCU_DEP_AMORT,5246,57.152195
NET_TOT_PPE,5246,57.152195
CONST_IN_PROGRESS,5246,57.152195
TOT_INV_AND_OTH_ASSETS,5246,57.152195
TOT_INTANGIBLE_ASSETS,5246,57.152195


### Missing-Value Decision

Missing values were retained rather than automatically replaced with zero or
removed. The high levels of missingness appear to be related to differences
in variables reported across the study period. Missing-value treatment for
specific modeling variables will be considered during preprocessing.

In [11]:
# Keep the original missing values for now.
# Missing-value treatment for individual modeling features will be handled
# during preprocessing after the EDA identifies the variables being used.

print("Total missing values:", df.isna().sum().sum())

Total missing values: 194694


## 9. Standardize Data Types

Standardize text fields and convert the reporting date fields to datetime.
`YEAR_QTR` is retained as a string because it identifies the reporting period.

In [12]:
# Remove leading/trailing spaces from text fields
text_columns = df.select_dtypes(include="object").columns

for column in text_columns:
    df[column] = df[column].str.strip()

# Convert reporting dates to datetime
df["BEG_DATE"] = pd.to_datetime(df["BEG_DATE"], errors="coerce")
df["END_DATE"] = pd.to_datetime(df["END_DATE"], errors="coerce")

# Keep YEAR_QTR as a reporting-period identifier
df["YEAR_QTR"] = df["YEAR_QTR"].astype(str)

print("Text and date fields standardized.")

Text and date fields standardized.


In [13]:
# Verify the converted fields
display(df[["YEAR_QTR", "BEG_DATE", "END_DATE"]].head())

,YEAR_QTR,BEG_DATE,END_DATE
0,20211,2021-01-01,2021-03-31
1,20211,2021-01-01,2021-03-31
2,20211,2021-01-01,2021-03-31
3,20211,2021-01-01,2021-03-31
4,20211,2021-01-01,2021-03-31


## 10. Investigate Negative Values

Negative values were found in several numeric variables. They were not
automatically removed because some financial and accounting measures can
legitimately have negative values.

Negative values will be retained unless the HCAI variable definition indicates
that they are invalid.

In [14]:
# Identify numeric columns
numeric_columns = df.select_dtypes(include=np.number).columns

# Count negative values in each numeric column
negative_values = (df[numeric_columns] < 0).sum()

display(
    negative_values[negative_values > 0]
    .sort_values(ascending=False)
)

DISP_855                    2390
NONOP_REV                   1012
TOT_EQUITY                   692
NET_OTH                      560
BAD_DEBT                     420
NET_INDGNT                   202
CADJ_MCAL                    160
NET_MCAL                     139
SUB_INDGNT                   131
DED_OTH                      103
NET_THRD                      99
CADJ_MCAL_MC                  90
OTH_OP_REV                    65
TCH_SUPP                      64
NET_CNTY                      64
CADJ_THRD                     61
NET_MCAL_MC                   57
DED_TOT                       49
NET_THRD_MC                   49
CAP_TOT                       46
CAP_THRD                      38
NET_MCAR                      22
CAP_MCAR                      20
CHAR_OTH                      20
NET_MCAR_MC                   19
TOT_INV_AND_OTH_ASSETS        18
CADJ_MCAR_MC                  17
CADJ_THRD_MC                  17
CADJ_MCAR                     17
CADJ_CNTY                     14
CAP_MCAL  

### Result

Negative values were found in several financial and adjustment variables.
They were retained because negative amounts can be valid for certain financial
measures, such as equity, adjustments, and deductions.

## 11. Validate Hospital Bed Counts

Check important hospital capacity variables for potentially inconsistent values.

In [15]:
# Review basic statistics for hospital capacity and utilization variables
key_columns = [
    "LIC_BEDS",
    "AVL_BEDS",
    "STF_BEDS",
    "DIS_TOT",
    "DAY_TOT",
    "VIS_TOT"
]

display(df[key_columns].describe().T)

,count,mean,std,min,25%,50%,75%,max
LIC_BEDS,9179.0,219.515089,223.562491,0.0,66.5,152.0,317.5,1500.0
AVL_BEDS,9179.0,205.257762,202.600614,0.0,62.0,148.0,291.0,1459.0
STF_BEDS,9179.0,162.748339,182.109408,0.0,49.0,110.0,223.0,1459.0
DIS_TOT,9179.0,1912.889313,2123.838658,0.0,242.0,1057.0,2948.0,16454.0
DAY_TOT,9179.0,12623.575335,15231.205107,0.0,3386.0,7848.0,16825.0,131028.0
VIS_TOT,9179.0,32744.002941,60832.702136,0.0,2346.5,14734.0,38537.0,771674.0


In [16]:
# Identify records where available or staffed beds exceed licensed beds
if all(column in df.columns for column in ["LIC_BEDS", "AVL_BEDS", "STF_BEDS"]):
    
    invalid_beds = df[
        (df["AVL_BEDS"] > df["LIC_BEDS"]) |
        (df["STF_BEDS"] > df["LIC_BEDS"])
    ]
    
    print("Records with potentially inconsistent bed counts:",
          len(invalid_beds))

Records with potentially inconsistent bed counts: 120


### Result

A total of 120 records were identified for further investigation. These records
were retained because the differences may reflect reporting definitions or
operational circumstances and were not confirmed as data errors.

## 12. Check for Outliers

Use the IQR method to identify unusually high or low numeric values. Outliers
are not automatically removed because large hospitals may legitimately have
higher utilization and financial values than smaller facilities.

In [28]:
# Identify potential IQR outliers in numeric variables
outlier_summary = []

for column in numeric_columns:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    count = ((df[column] < lower) | (df[column] > upper)).sum()

    outlier_summary.append({
        "column": column,
        "outlier_count": count,
        "outlier_percent": count / len(df) * 100
    })

outliers = (
    pd.DataFrame(outlier_summary)
    .sort_values("outlier_count", ascending=False)
)

display(outliers.head(20))

,column,outlier_count,outlier_percent
67,DISP_855,2266,24.686785
83,CAP_TOT,2163,23.564658
92,NET_INDGNT,2016,21.963177
98,NONOP_REV,1971,21.472927
97,PHY_COMP,1761,19.185096
48,GRIP_INDGNT,1595,17.376621
59,GROP_INDGNT,1548,16.864582
82,CAP_THRD,1533,16.701166
88,NET_CNTY,1433,15.611722
95,OTH_OP_REV,1415,15.415623


### Outlier Decision

The IQR method identified many potential outliers. These observations were
retained because extreme values may represent legitimate differences between
large and small hospitals rather than data errors. Outlier treatment can be
considered during exploratory analysis and modeling.

## 13. Remove Exact Duplicates

No exact duplicate rows or duplicate facility-quarter records were found.
The following step is included as a final safeguard.

In [29]:
# Remove completely identical rows, if any
df = df.drop_duplicates().reset_index(drop=True)

print("Shape after removing exact duplicates:", df.shape)

Shape after removing exact duplicates: (9179, 159)


## 14. Final Data Quality Check

Review the cleaned dataset before saving it for exploratory data analysis.

In [19]:
print("Final dataset shape:", df.shape)
print("Total missing values:", df.isna().sum().sum())
print("Total exact duplicate rows:", df.duplicated().sum())
print("Reporting periods:", df["YEAR_QTR"].min(), "to", df["YEAR_QTR"].max())

Final dataset shape: (9179, 159)
Total missing values: 194695
Total exact duplicate rows: 0
Reporting periods: 20211 to 20261


## 15. Save the Clean Dataset

Save the cleaned combined dataset so it can be used in the exploratory data
analysis and preprocessing notebooks.

In [30]:
# Create the processed-data folder if needed
processed_path = Path("../data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

# Save the cleaned dataset
output_file = processed_path / "hcai_hospital_data_clean.csv"
df.to_csv(output_file, index=False)

print(f"Clean dataset saved to: {output_file}")

Clean dataset saved to: ../data/processed/hcai_hospital_data_clean.csv


## Conclusion

The 21 quarterly HCAI datasets were successfully combined into a single dataset
containing 9,179 hospital-quarter records and 159 variables covering 2021 Q1
through 2026 Q1.

Data quality checks found no exact duplicate rows and no duplicate
facility-quarter records. The analysis also identified substantial missing
values associated with differences in reporting periods and the HCAI schema.

Potentially inconsistent bed counts, negative financial values, and statistical
outliers were identified but were not automatically removed because they may
represent valid hospital or financial observations.

Text fields were standardized and reporting date fields were converted to
datetime format. The resulting dataset was saved for the next stage of the
project.